# 03 — PPO training

Обучение использует только train seed 1000–9999. Validation callback выбирает лучший checkpoint на фиксированных validation seed; test set здесь недоступен.

In [ ]:
from pathlib import Path
import sys
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/gas_portfolio_rl')
except ImportError:
    IN_COLAB = False
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd if (cwd / 'pyproject.toml').exists() else cwd / 'gas_portfolio_rl'
sys.path.insert(0, str(PROJECT_ROOT))
from notebook_bootstrap import bootstrap_project
PROJECT_ROOT = bootstrap_project(PROJECT_ROOT, install_dependencies=IN_COLAB, install_rl_dependencies=True)
import gas_portfolio_rl
print('Project:', PROJECT_ROOT)
print('Package:', gas_portfolio_rl.__file__)

In [ ]:
from dataclasses import replace
from datetime import datetime
from gas_portfolio_rl import AppConfig
from gas_portfolio_rl.config import resolve_calibrated_market
from gas_portfolio_rl.training import train_ppo
from gas_portfolio_rl.visualization import plot_training_diagnostics
from IPython.display import display

QUICK_RUN = True  # False: полный профиль (v5_full = 600,000 PPO decisions); True: smoke-run.
MODEL_PROFILE = 'ppo_v5_full.yaml'  # Для ablation: control, long_horizon или stable.
TRAINING_SEED = 1000
CALIBRATED_MARKET = resolve_calibrated_market(PROJECT_ROOT)
config = AppConfig.from_yaml(
    CALIBRATED_MARKET,
    PROJECT_ROOT / 'configs/environment/default.yaml',
    PROJECT_ROOT / 'configs/reward/default.yaml',
    PROJECT_ROOT / 'configs/baselines/default.yaml',
    PROJECT_ROOT / 'configs/evaluation/default.yaml',
    PROJECT_ROOT / 'configs/models' / MODEL_PROFILE,
)
if QUICK_RUN:
    config = replace(config, ppo=replace(config.ppo, total_timesteps=4096, n_envs=2, n_steps=128, batch_size=128, n_epochs=2, eval_frequency=2048, validation_episodes=2, validation_stress_episodes=1, checkpoint_frequency=2048))
RUN_DIR = PROJECT_ROOT / 'results/ppo' / config.ppo.experiment_tag / datetime.now().strftime('run_%Y%m%d_%H%M%S')
print('Market config:', CALIBRATED_MARKET)
print('Model profile:', MODEL_PROFILE)
print('Experiment tag:', config.ppo.experiment_tag)
print('Run directory:', RUN_DIR)
print('PPO:', config.ppo)

In [ ]:
result = train_ppo(config, RUN_DIR, training_seed=TRAINING_SEED, live_display=True)
print('Best validation score:', result.best_validation_score)
print('Best model:', result.best_model_path)
print('Best normalization:', result.best_normalization_path)
display(plot_training_diagnostics(RUN_DIR))

In [ ]:
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(RUN_DIR), f'{path.stat().st_size / 1024:.1f} KiB')